# Staging Models

**Layer:** staging (raw files -> clean tables)

Each staging model maps **one raw source to one table** and does only four things:
1. **Rename** columns to clear snake_case names
2. **Cast** every column to an explicit type
3. **Recode** survey missing-value codes (0, 8, 9, -8, -9, blanks) to `NULL`
4. **Label** numeric response codes so a reader never has to open a codebook

## Models Built Here

| Model | Source file | Grain |
|---|---|---|
| `stg_ces` | `cumulative_2006-2025_zstd.feather` (converted to parquet first) | respondent |
| `stg_anes` | `anes_cdf.parquet` | respondent |
| `stg_pew` | `pew_economy_rating_by_party.csv` | poll |
| `stg_umich_party` | `consumer_sentiment_by_party.csv` | month |
| `stg_umich_index` | `umich_consumer_sentiment_index_data.csv` | month × measure × subgroup (long) |
| `stg_inflation` | `monthly_inflation_rates_2000_2026.csv` | month |
| `stg_cpi` | `cpi_data.csv` | month |
| `stg_misery_index` | `misery_index.csv` | month |
| `stg_fred_observations` | 11 FRED files (rates, markets, saving, PCE, earnings, debt service) | series × date (long) |
| `ref_events` | `inauguration_election_dates.csv` | presidential transition |

# 0. Setup
Sets the catalog/schema so every table name below is unqualified, pins the session time zone (CES start times are
UTC, and `to_date()` uses the session time zone), and creates one new volume, `converted_data`, to hold the CES
file after conversion. Keeping it out of `raw_data` keeps that volume an untouched copy of the GitHub `data/` folder.


In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA public_opinion_economy_case_study;

-- Make date extraction from timestamps deterministic: every to_date(timestamp) below is a UTC date.
SET TIME ZONE 'UTC';

CREATE VOLUME IF NOT EXISTS converted_data
COMMENT 'Machine-converted copies of raw files that Spark cannot read natively (e.g. CES feather -> parquet). Never edited by hand.';

## 1. One-time conversion: CES feather -> parquet (Python)
Spark has no Feather reader, so `read_files()` cannot open the CES file directly. This cell converts it once with
`pyarrow` (pre-installed on Databricks). It is a format change only: every row and column is carried over unchanged,
and the row/column counts are asserted so a silent truncation would fail loudly.

Writes go to local `/tmp` first and are then copied into the volume, because volumes support sequential writes only
and a single copy is the safest way to land a finished file.

In [0]:
%python
import shutil
import pyarrow.feather as feather
import pyarrow.parquet as pq

SRC = "/Volumes/workspace/public_opinion_economy_case_study/raw_data/cumulative_2006-2025_zstd.feather"
DST = "/Volumes/workspace/public_opinion_economy_case_study/converted_data/ces_cumulative_2006_2025.parquet"
TMP = "/tmp/ces_cumulative_2006_2025.parquet"

table = feather.read_table(SRC)                 # Feather v2 with zstd compression
pq.write_table(table, TMP, compression="zstd")  # same columns, same types; labeled factors stay strings
shutil.copyfile(TMP, DST)

# Guardrails: the cumulative file has 718,955 rows x 109 columns (2006-2025).
check = pq.read_metadata(DST)
assert (check.num_rows, check.num_columns) == (table.num_rows, table.num_columns) == (718_955, 109), \
    f"Unexpected shape: {check.num_rows} rows x {check.num_columns} cols"
print(f"Converted {check.num_rows:,} rows x {check.num_columns} columns -> {DST}")

## 2. `stg_ces` — Cooperative Election Study, cumulative 2006–2025
**Grain:** one row per respondent (`respondent_key` = year + case_id; case_id is unique within each year).
**Rows:** 718,955 (all years kept; the 2016–2025 window is applied downstream).

Things found in the data that this model handles or documents:
* **Party codes differ from ANES.** CES `pid3_leaner` is 1 = Dem, **2 = Rep, 3 = Ind**, 8 = Not sure. ANES `VCF0303`
  is 1 = Dem, **2 = Ind, 3 = Rep**. Labeling both here to the same text prevents the classic swap bug.
* **`economy_retro` code 6 = "Not sure"** is a real answer, not missing, so it keeps its label. Whether it sits in the
  denominator is an analysis decision for `int_respondents` (see the QA cell at the end: it moves the headline by
  about 2 points).
* **`investor` was not asked in 2023 or 2025** (per the CES guide), so it is NULL for every respondent in those years.
  It cannot be used as a control in the 2024→2025 comparison.
* `weight` is the year-specific weight (use it for every within-year estimate); `weight_cumulative` rescales years to
  equal size and is only for pooled multi-year estimates. Weights are complete for 2016–2025; 141 respondents in
  2010 have a weight of 0, which matters only if the CES window is extended back for the 2008–09 transition.

In [0]:
%sql
CREATE OR REPLACE TABLE stg_ces
COMMENT 'CES cumulative file 2006-2025, one row per respondent. Staging: renamed, typed, codes labeled, missing codes set to NULL. No filters or analysis variables.'
AS
WITH source AS (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/public_opinion_economy_case_study/converted_data/ces_cumulative_2006_2025.parquet',
    format => 'parquet'
  )
),

renamed AS (
  SELECT
    -- ── Keys ──────────────────────────────────────────────────────────────
    CAST(year AS INT)                                                    AS survey_year,
    CAST(case_id AS BIGINT)                                              AS case_id,      -- stored as double in source; all values are whole numbers
    concat_ws('-', 'ces', CAST(year AS STRING), CAST(CAST(case_id AS BIGINT) AS STRING)) AS respondent_key,

    -- ── Weights ───────────────────────────────────────────────────────────
    CAST(weight AS DOUBLE)                                               AS weight,             -- year-specific (primary)
    CAST(weight_cumulative AS DOUBLE)                                    AS weight_cumulative,  -- pooled multi-year only

    -- ── Fieldwork timing ──────────────────────────────────────────────────
    -- Pre-election wave start time. Even years field Sep-Nov (during the campaign), odd years Nov-Dec.
    -- int_econ_windows uses the median start date per year to align the 3-month economic window.
    starttime                                                            AS start_ts_utc,
    to_date(starttime)                                                   AS start_date_utc,
    CASE CAST(tookpost AS INT) WHEN 1 THEN TRUE WHEN 0 THEN FALSE END    AS took_post_wave,  -- even years only; NULL in odd years

    -- ── Geography (kept as strings to preserve leading zeros) ─────────────
    st                                                                   AS state_abbr,
    state                                                                AS state_name,
    county_fips,                                                                             -- 5-char string, joins to BLS LAUS
    zipcode,                                                                                 -- 5-char string, joins to Imprint ZIP aggregates

    -- ── Party identification ──────────────────────────────────────────────
    -- 3-category WITH leaners grouped into parties (Pew convention, used in every source).
    CAST(pid3_leaner AS INT)                                             AS pid3_leaner_code,
    CASE CAST(pid3_leaner AS INT)
      WHEN 1 THEN 'Democrat'
      WHEN 2 THEN 'Republican'      -- NOTE: 2 = Republican in CES (2 = Independent in ANES)
      WHEN 3 THEN 'Independent'     -- pure independents only; leaners are already in the parties
      WHEN 8 THEN 'Not sure'
    END                                                                  AS party3,

    -- 7-point scale: lets a robustness check move leaners back to "Independent" (codes 3 and 5).
    -- Code 9 appears for 3 respondents in 2021 and is undocumented, so it is set to NULL.
    CASE WHEN CAST(pid7 AS INT) BETWEEN 1 AND 8 THEN CAST(pid7 AS INT) END AS pid7_code,
    CASE CAST(pid7 AS INT)
      WHEN 1 THEN 'Strong Democrat'
      WHEN 2 THEN 'Not very strong Democrat'
      WHEN 3 THEN 'Lean Democrat'
      WHEN 4 THEN 'Independent'
      WHEN 5 THEN 'Lean Republican'
      WHEN 6 THEN 'Not very strong Republican'
      WHEN 7 THEN 'Strong Republican'
      WHEN 8 THEN 'Not sure'
    END                                                                  AS pid7,

    ideo5                                                                AS ideology5,  -- already text in source

    -- ── Outcome: retrospective national economy ───────────────────────────
    -- "OVER THE PAST YEAR the nation's economy has...?"  The plan's primary binary
    -- (better = codes 1-2, worse = codes 4-5) is built in int_respondents, not here.
    CAST(economy_retro AS INT)                                           AS economy_retro_code,
    CASE CAST(economy_retro AS INT)
      WHEN 1 THEN 'Gotten much better'
      WHEN 2 THEN 'Gotten somewhat better'
      WHEN 3 THEN 'Stayed about the same'
      WHEN 4 THEN 'Gotten somewhat worse'
      WHEN 5 THEN 'Gotten much worse'
      WHEN 6 THEN 'Not sure'
    END                                                                  AS economy_retro,

    -- ── Presidential approval (useful for the co-partisan story) ──────────
    CAST(approval_pres AS INT)                                           AS approval_pres_code,
    CASE CAST(approval_pres AS INT)
      WHEN 1 THEN 'Strongly approve'
      WHEN 2 THEN 'Somewhat approve'
      WHEN 3 THEN 'Somewhat disapprove'
      WHEN 4 THEN 'Strongly disapprove'
      WHEN 5 THEN 'Never heard / not sure'
      WHEN 6 THEN 'Neither approve nor disapprove'   -- 2007 only
    END                                                                  AS approval_pres,

    CASE CAST(newsint AS INT)
      WHEN 1 THEN 'Most of the time'
      WHEN 2 THEN 'Some of the time'
      WHEN 3 THEN 'Only now and then'
      WHEN 4 THEN 'Hardly at all'
      WHEN 7 THEN 'Do not know'
    END                                                                  AS news_interest,

    -- ── Demographics (controls X in the RQ1 model) ────────────────────────
    CAST(age AS INT)                                                     AS age,
    CAST(birthyr AS INT)                                                 AS birth_year,
    CASE CAST(gender AS INT) WHEN 1 THEN 'Male' WHEN 2 THEN 'Female' END AS sex,  -- "gender" in CES = standardized sex item

    CAST(educ AS INT)                                                    AS educ_code,
    CASE CAST(educ AS INT)
      WHEN 1 THEN 'No HS'
      WHEN 2 THEN 'High school graduate'
      WHEN 3 THEN 'Some college'
      WHEN 4 THEN '2-year degree'
      WHEN 5 THEN '4-year degree'
      WHEN 6 THEN 'Post-grad'
    END                                                                  AS education,

    CASE CAST(race_h AS INT)        -- race with any-part Hispanic coded as Hispanic
      WHEN 1 THEN 'White'
      WHEN 2 THEN 'Black'
      WHEN 3 THEN 'Hispanic'
      WHEN 4 THEN 'Asian'
      WHEN 5 THEN 'Native American'
      WHEN 6 THEN 'Mixed'
      WHEN 7 THEN 'Other'
      WHEN 8 THEN 'Middle Eastern'
    END                                                                  AS race_ethnicity,

    -- ── Economic circumstances ────────────────────────────────────────────
    -- faminc arrives as text brackets. Keep the label (incl. "Prefer not to say", 9.5% of 2016+ respondents,
    -- which int_respondents can model as its own category) and add an ordinal rank for ordered use.
    CASE WHEN faminc = 'Skipped' THEN NULL ELSE faminc END               AS family_income,
    CASE faminc
      WHEN 'Less than 10k' THEN 1  WHEN '10k - 20k'   THEN 2  WHEN '20k - 30k'   THEN 3
      WHEN '30k - 40k'     THEN 4  WHEN '40k - 50k'   THEN 5  WHEN '50k - 60k'   THEN 6
      WHEN '60k - 70k'     THEN 7  WHEN '70k - 80k'   THEN 8  WHEN '80k - 100k'  THEN 9
      WHEN '100k - 120k'   THEN 10 WHEN '120k - 150k' THEN 11 WHEN '150k+'       THEN 12
    END                                                                  AS family_income_rank,  -- NULL for prefer-not-to-say / skipped

    employ                                                               AS employment_status,   -- text in source (Full-Time, Retired, ...)
    ownhome                                                              AS home_ownership,      -- Own / Rent / Other
    investor                                                             AS stock_investor,      -- Yes / No; NOT ASKED in 2023 or 2025
    CASE CAST(`union` AS INT)        -- backticks: UNION is a reserved word
      WHEN 1 THEN 'Yes, currently'
      WHEN 2 THEN 'Yes, formerly'
      WHEN 3 THEN 'No, never'
    END                                                                  AS union_member,
    has_child                                                            AS has_child_under_18
  FROM source
)

SELECT * FROM renamed;

## 3. `stg_anes` — ANES Time Series Cumulative Data File
**Grain:** one row per respondent per study year (1948–2024; 73,745 rows). Only the ~25 variables the plan uses are
kept out of 1,030.

Things found in the data that this model handles or documents:
* **The two outcome items use different code schemes.** `VCF0870` (national economy) is **1 / 3 / 5**
  (better / same / worse); `VCF0880` (own finances) is **1 / 2 / 3**. Treating them alike would silently code every
  "worse" on the national item as missing. Both are labeled to the same text here.
* **Most columns are strings with blanks** (`' '` = question not used that year, e.g. 245 blanks on VCF0870 in 2024),
  so every value is trimmed and `try_cast` to INT before recoding.
* **Plan correction — no strata or PSU in this file.** The plan names `VCF0010z` (strata) and `VCF0011z` (PSU), but
  the codebook lists both as **weights** (1970 type-1/type-2 weights; full-sample weights for 2012–2024), and their
  values are continuous weights. The cumulative file does not carry design variables. Design-based SEs for RQ3 need
  the strata/PSU merged in from each year's full study file (2016, 2020, 2024) on case ID, which is a new data gap.
  Until then RQ3 uses `weight` with a bootstrap.
* **Control-variable coverage problems** worth knowing before RQ3 modeling: home ownership (`VCF0146`) is "NA" for
  51% of 2020 respondents, and state (`VCF0901a`) is blank for 44% of 2024 respondents. Age is top-coded at 90 in
  2016 and at 80 in 2020 and 2024.

In [0]:
CREATE OR REPLACE TABLE stg_anes
COMMENT 'ANES cumulative data file, one row per respondent-year (1948-2024), plan variables only. Staging: renamed, typed, codes labeled, missing codes set to NULL. No strata/PSU exist in this file.'
AS
WITH source AS (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/public_opinion_economy_case_study/raw_data/anes_cdf.parquet',
    format => 'parquet'
  )
),

-- Step 1: trim + try_cast every coded variable to INT once, so the labeling step below stays readable.
typed AS (
  SELECT
    CAST(VCF0004 AS INT)                        AS survey_year,
    CAST(VCF0006 AS BIGINT)                     AS case_id,          -- year-level case ID (pre-election case ID)
    CAST(VCF0009z AS DOUBLE)                    AS weight,           -- full-sample weight 2012-2024 (mean 1 within year)
    CAST(VCF0017 AS INT)                        AS mode_code,
    CAST(VCF0013 AS INT)                        AS post_complete_code,
    try_cast(trim(VCF0870)  AS INT)             AS v0870,  -- national economy, past year (1/3/5)
    try_cast(trim(VCF0871)  AS INT)             AS v0871,  -- national economy, past year, 5-point
    try_cast(trim(VCF0872)  AS INT)             AS v0872,  -- national economy, next year (1/3/5)
    try_cast(trim(VCF0880)  AS INT)             AS v0880,  -- own finances, past year (1/2/3)
    try_cast(trim(VCF0880a) AS INT)             AS v0880a, -- own finances, past year, 5-point
    try_cast(trim(VCF0881)  AS INT)             AS v0881,  -- own finances, next year (1/2/3)
    try_cast(trim(VCF0301)  AS INT)             AS v0301,  -- party ID, 7-point
    try_cast(trim(VCF0303)  AS INT)             AS v0303,  -- party ID, 3-category
    try_cast(trim(VCF9009)  AS INT)             AS v9009,  -- approve of president on the economy
    try_cast(trim(VCF0101)  AS INT)             AS v0101,  -- age
    CAST(VCF0104 AS INT)                        AS v0104,  -- gender
    try_cast(trim(VCF0105b) AS INT)             AS v0105b, -- race-ethnicity, 4 categories
    CAST(VCF0110 AS INT)                        AS v0110,  -- education, 4 categories
    try_cast(trim(VCF0114)  AS INT)             AS v0114,  -- family income percentile group
    try_cast(trim(VCF0118)  AS INT)             AS v0118,  -- work status, 5 categories
    try_cast(trim(VCF0146)  AS INT)             AS v0146,  -- home ownership
    try_cast(trim(VCF0156)  AS INT)             AS v0156,  -- laid off in last 6 months
    try_cast(trim(VCF9224)  AS INT)             AS v9224,  -- has money in stock market
    nullif(trim(VCF0901a), '')                  AS v0901a  -- state FIPS
  FROM source
)

-- Step 2: label. Valid codes get text; DK keeps a label (it is an answer); NA / INAP / 0 codes become NULL.
SELECT
  -- ── Keys, weight, admin ─────────────────────────────────────────────────
  survey_year,
  case_id,
  concat_ws('-', 'anes', CAST(survey_year AS STRING), CAST(case_id AS STRING))  AS respondent_key,
  weight,
  CASE mode_code
    WHEN 0 THEN 'All personal (face-to-face)'
    WHEN 1 THEN 'Telephone pre, personal or no post'
    WHEN 2 THEN 'Personal pre, telephone post'
    WHEN 3 THEN 'All telephone'
    WHEN 4 THEN 'All internet'
    WHEN 5 THEN 'All video (2020)'
    WHEN 6 THEN 'Personal pre, video post (2024)'
    WHEN 7 THEN 'Personal pre, telephone post (2024)'
    WHEN 8 THEN 'Telephone pre, personal post (2024)'
    WHEN 9 THEN 'Telephone pre, video post (2024)'
  END                                                         AS interview_mode,   -- for the mode-change robustness check
  post_complete_code = 1                                      AS has_post_interview,

  -- ── National economy (RQ3 numerator: "the economy") ────────────────────
  CASE WHEN v0870 IN (1, 3, 5, 8) THEN v0870 END              AS econ_national_retro_code,
  CASE v0870 WHEN 1 THEN 'Better' WHEN 3 THEN 'Stayed the same' WHEN 5 THEN 'Worse' WHEN 8 THEN 'Do not know' END
                                                              AS econ_national_retro,
  CASE v0871 WHEN 1 THEN 'Much better' WHEN 2 THEN 'Somewhat better' WHEN 3 THEN 'Stayed the same'
             WHEN 4 THEN 'Somewhat worse' WHEN 5 THEN 'Much worse' WHEN 8 THEN 'Do not know' END
                                                              AS econ_national_retro_5pt,   -- 0, 9, -2, -4 -> NULL
  CASE v0872 WHEN 1 THEN 'Get better' WHEN 3 THEN 'Stay the same' WHEN 5 THEN 'Get worse' WHEN 8 THEN 'Do not know' END
                                                              AS econ_national_prosp,       -- 9 (NA) -> NULL

  -- ── Own finances (RQ3 comparison: "your wallet") ───────────────────────
  CASE WHEN v0880 IN (1, 2, 3, 9) THEN v0880 END              AS personal_fin_retro_code,   -- 0 (NA / no post IW) -> NULL
  CASE v0880 WHEN 1 THEN 'Better' WHEN 2 THEN 'Stayed the same' WHEN 3 THEN 'Worse' WHEN 9 THEN 'Do not know' END
                                                              AS personal_fin_retro,        -- NOTE: 9 is DK here, not NA
  CASE v0880a WHEN 1 THEN 'Much better' WHEN 2 THEN 'Somewhat better' WHEN 3 THEN 'Stayed the same'
              WHEN 4 THEN 'Somewhat worse' WHEN 5 THEN 'Much worse' WHEN 8 THEN 'Do not know' END
                                                              AS personal_fin_retro_5pt,    -- 9 (NA) -> NULL
  CASE v0881 WHEN 1 THEN 'Better' WHEN 2 THEN 'Stay the same' WHEN 3 THEN 'Worse' WHEN 9 THEN 'Do not know' END
                                                              AS personal_fin_prosp,        -- 0 (NA) -> NULL

  -- ── Party identification ───────────────────────────────────────────────
  CASE WHEN v0303 IN (1, 2, 3) THEN v0303 END                 AS party3_code,
  CASE v0303
    WHEN 1 THEN 'Democrat'          -- includes leaners
    WHEN 2 THEN 'Independent'       -- NOTE: 2 = Independent in ANES (2 = Republican in CES)
    WHEN 3 THEN 'Republican'        -- includes leaners
  END                                                         AS party3,
  CASE WHEN v0301 BETWEEN 1 AND 7 THEN v0301 END              AS pid7_code,
  CASE v0301
    WHEN 1 THEN 'Strong Democrat'   WHEN 2 THEN 'Weak Democrat'   WHEN 3 THEN 'Independent - Democrat'
    WHEN 4 THEN 'Independent'
    WHEN 5 THEN 'Independent - Republican' WHEN 6 THEN 'Weak Republican' WHEN 7 THEN 'Strong Republican'
  END                                                         AS pid7,

  CASE v9009
    WHEN 1 THEN 'Approve strongly'    WHEN 2 THEN 'Approve not strongly'
    WHEN 4 THEN 'Disapprove not strongly' WHEN 5 THEN 'Disapprove strongly'
    WHEN 8 THEN 'Do not know'
  END                                                         AS approve_president_economy,

  -- ── Demographics ───────────────────────────────────────────────────────
  CASE WHEN v0101 >= 17 THEN v0101 END                        AS age,  -- 0 = NA; top-coded 90 (2016), 80 (2020, 2024)
  CASE v0104 WHEN 1 THEN 'Male' WHEN 2 THEN 'Female' WHEN 3 THEN 'Other' END AS gender,  -- 'Other' offered 2016 only
  CASE v0105b
    WHEN 1 THEN 'White non-Hispanic' WHEN 2 THEN 'Black non-Hispanic'
    WHEN 3 THEN 'Hispanic'           WHEN 4 THEN 'Other or multiple, non-Hispanic'
  END                                                         AS race_ethnicity,
  CASE WHEN v0110 BETWEEN 1 AND 4 THEN v0110 END              AS educ_code,
  CASE v0110
    WHEN 1 THEN 'Grade school or less' WHEN 2 THEN 'High school'
    WHEN 3 THEN 'Some college'         WHEN 4 THEN 'College or advanced degree'
  END                                                         AS education,

  -- ── Economic circumstances (RQ3 controls) ──────────────────────────────
  -- Income is stored as national percentile bands, which keeps it comparable across years with different brackets.
  CASE WHEN v0114 BETWEEN 1 AND 5 THEN v0114 END              AS income_pctile_code,
  CASE v0114
    WHEN 1 THEN '0-16th percentile'  WHEN 2 THEN '17-33rd percentile' WHEN 3 THEN '34-67th percentile'
    WHEN 4 THEN '68-95th percentile' WHEN 5 THEN '96-100th percentile'
  END                                                         AS income_pctile,
  CASE v0118
    WHEN 1 THEN 'Employed'  WHEN 2 THEN 'Not employed (laid off, unemployed, disabled, other)'
    WHEN 3 THEN 'Retired'   WHEN 4 THEN 'Homemaker'  WHEN 5 THEN 'Student'
  END                                                         AS work_status,
  CASE v0146 WHEN 1 THEN 'Own' WHEN 2 THEN 'Do not own' END   AS home_ownership,  -- 51% NA in 2020
  CASE v0156
    WHEN 1 THEN 'Yes'
    WHEN 5 THEN 'No'
    WHEN 0 THEN 'Not currently working (not asked)'  -- a skip pattern, not missing data
  END                                                         AS laid_off_last_6mo,
  CASE v9224 WHEN 1 THEN 'Yes' WHEN 2 THEN 'No' END           AS stock_investor,   -- -8 DK, -9 NA -> NULL

  lpad(v0901a, 2, '0')                                        AS state_fips        -- 44% blank in 2024
FROM typed;

## 4. `stg_pew` — Pew economy ratings by party
**Grain:** one row per poll (28 polls, Nov 2016 – Oct 2026). Values are the % rating national economic conditions
excellent or good, with leaners grouped into parties. Toplines only: no sample sizes or SEs (a known data gap).

The source header contains `/` and spaces (`Rep/Lean Rep`), which are invalid in Delta column names, so the columns
are renamed here. The `Date` field is undocumented (fieldwork end vs. publication date): verify against Pew's
methodology pages before using it for event windows measured in weeks.

In [0]:
CREATE OR REPLACE TABLE stg_pew
COMMENT 'Pew Research: % rating the national economy excellent/good, total and by party incl. leaners. One row per poll. Toplines only, no SEs.'
AS
WITH source AS (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/public_opinion_economy_case_study/raw_data/pew_economy_rating_by_party.csv',
    format => 'csv', header => true, inferColumnTypes => false
  )
)
SELECT
  to_date(`Date`)                          AS poll_date,
  try_cast(Total          AS DOUBLE)       AS pct_excellent_good_total,
  try_cast(`Rep/Lean Rep` AS DOUBLE)       AS pct_excellent_good_rep,
  try_cast(`Dem/Lean Dem` AS DOUBLE)       AS pct_excellent_good_dem
FROM source;

## 5. `stg_umich_party` — UMich Index of Consumer Sentiment by party
**Grain:** one row per month (113 months, Apr 2017 – Aug 2026, no gaps). Primary RQ1 series.
Kept wide (three party columns) because every downstream use compares the parties side by side.

In [0]:
CREATE OR REPLACE TABLE stg_umich_party
COMMENT 'University of Michigan Surveys of Consumers: Index of Consumer Sentiment within party, monthly Apr 2017 - Aug 2026. No SEs in source.'
AS
WITH source AS (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/public_opinion_economy_case_study/raw_data/consumer_sentiment_by_party.csv',
    format => 'csv', header => true, inferColumnTypes => false
  )
)
SELECT
  to_date(`date`)                      AS month,              -- first of month
  try_cast(republican  AS DOUBLE)      AS sentiment_rep,
  try_cast(democrat    AS DOUBLE)      AS sentiment_dem,
  try_cast(independent AS DOUBLE)      AS sentiment_ind
FROM source;

## 6. `stg_umich_index` — UMich index file, unpivoted to long format
**Grain:** one row per month × measure × subgroup. **Rows:** 92,158 (128 months × 720 value columns, minus 2 blank
cells: Feb 2016 `hompx1_r50_ne` and Nov 2024 `hompx5_r50_ehs`).

**Why long:** the source is 721 columns wide. Every column name is `<measure>_<subgroup>` (45 measures × 16
subgroups), so splitting the name gives two clean dimensions. Long format means a mart asks for
`measure_code = 'px1_med' AND subgroup_code = 'all'` instead of hard-coding column names, and a new column added by
UMich flows through without editing this model.

**How the unpivot works without listing 720 columns:** pack each row into a JSON object (`to_json(struct(*))`),
parse it back as a `map<column_name, value>`, and `explode` the map into rows (two CTEs, because Spark does not allow `*` directly inside `explode`). Because CSVs are read as strings, all
values share one type and fit in one map; blank cells are dropped by `to_json` automatically.

**Units differ by measure, so each row carries a `unit`:**
* `index` – ICS / ICE / ICC (1966 = 100)
* `relative score` – % favorable − % unfavorable + 100 (so 100 = balanced)
* `net percent` – % citing one reason − % citing its opposite
* `percent` – expected price change (`px1_*`, `px5_*`, home prices). The plan's RQ2 primary measure is `px1_med`.
* `cents per gallon` – **gas expectations are expected change in cents, not percent**, so RQ2's gas comparison must
  convert them (e.g., divide by the current pump price) before setting them against the gasoline CPI.

In [0]:
CREATE OR REPLACE TABLE stg_umich_index
COMMENT 'University of Michigan Surveys of Consumers index file (Jan 2016 - Aug 2026) unpivoted to month x measure x demographic subgroup. No party split.'
AS
WITH source AS (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/public_opinion_economy_case_study/raw_data/umich_consumer_sentiment_index_data.csv',
    format => 'csv', header => true, inferColumnTypes => false
  )
),

-- Wide -> long, step 1: pack each whole row into one JSON object: {"yyyymm":"201601","ics_all":"92",...}
packed AS (
  SELECT
    yyyymm,
    to_json(struct(*))   AS row_json      -- blank cells are NULL and to_json leaves them out
  FROM source
),

-- Step 2: parse the JSON back as a map<column_name, value> and explode it: one output row per column.
-- (Spark does not allow * directly inside explode(), which is why the packing happens in its own step.)
unpivoted AS (
  SELECT
    to_date(p.yyyymm, 'yyyyMM')        AS month,
    kv.variable_name,
    try_cast(kv.raw_value AS DOUBLE)   AS value
  FROM packed AS p
  LATERAL VIEW explode(from_json(p.row_json, 'map<string,string>')) kv AS variable_name, raw_value
  WHERE kv.variable_name NOT IN ('yyyymm', '_rescued_data')   -- the date key and read_files' rescue column are not measures
),

-- Split "px1_mean_a1834" into measure "px1_mean" and subgroup "a1834".
-- The greedy (.+) runs to the LAST underscore, which is right because measure names contain underscores
-- but subgroup codes never do.
parsed AS (
  SELECT
    month,
    variable_name,
    regexp_extract(variable_name, '^(.+)_([a-z0-9]+)$', 1)   AS measure_code,
    regexp_extract(variable_name, '^(.+)_([a-z0-9]+)$', 2)   AS subgroup_code,
    value
  FROM unpivoted
),

-- Lookup: 45 measures, labeled from umich_consumer_sentiment_codebook.pdf.
measures AS (
  SELECT * FROM VALUES
    ('ics',         'Index of Consumer Sentiment',                                       'index'),
    ('ice',         'Index of Consumer Expectations',                                    'index'),
    ('icc',         'Index of Current Economic Conditions',                              'index'),
    ('pago_r',      'Personal finances vs a year ago',                                   'relative score'),
    ('pagorn_ny',   'Why finances changed: higher income minus lower income',            'net percent'),
    ('pexp_r',      'Expected personal finances a year from now',                        'relative score'),
    ('pago5_r',     'Personal finances vs 5 years ago',                                  'relative score'),
    ('pexp5_r',     'Expected personal finances 5 years from now',                       'relative score'),
    ('ptrd5_r',     'Five-year trend in personal finances (past and expected)',          'relative score'),
    ('news_r',      'Heard favorable vs unfavorable business news',                      'relative score'),
    ('newsrn_ne',   'News heard about employment: favorable minus unfavorable',          'net percent'),
    ('newsrn_ng',   'News heard about government/elections: favorable minus unfavorable','net percent'),
    ('newsrn_np',   'News heard about prices: favorable minus unfavorable',              'net percent'),
    ('umex_r',      'Unemployment next 12 months (less minus more)',                     'relative score'),
    ('ratex_r',     'Interest rates next 12 months (down minus up)',                     'relative score'),
    ('px1_mean',    'Expected price change next 12 months: mean',                        'percent'),
    ('px1_var',     'Expected price change next 12 months: variance',                    'percent squared'),
    ('px1_med',     'Expected price change next 12 months: median',                      'percent'),
    ('px1_std',     'Expected price change next 12 months: standard deviation',          'percent'),
    ('px1_p25',     'Expected price change next 12 months: 25th percentile',             'percent'),
    ('px1_p75',     'Expected price change next 12 months: 75th percentile',             'percent'),
    ('px1_r50',     'Expected price change next 12 months: interquartile range',         'percent'),
    ('px5_mean',    'Expected annual price change next 5-10 years: mean',                'percent'),
    ('px5_var',     'Expected annual price change next 5-10 years: variance',            'percent squared'),
    ('px5_med',     'Expected annual price change next 5-10 years: median',              'percent'),
    ('px5_std',     'Expected annual price change next 5-10 years: standard deviation',  'percent'),
    ('px5_p25',     'Expected annual price change next 5-10 years: 25th percentile',     'percent'),
    ('px5_p75',     'Expected annual price change next 5-10 years: 75th percentile',     'percent'),
    ('px5_r50',     'Expected annual price change next 5-10 years: interquartile range', 'percent'),
    ('govt_r',      'Government economic policy (good minus poor)',                      'relative score'),
    ('hom_r',       'Good vs bad time to buy a house',                                   'relative score'),
    ('homrn_ni',    'House-buying reason: good investment minus bad investment',         'net percent'),
    ('homrn_nt',    'House-buying reason: good times minus bad times',                   'net percent'),
    ('homrn_np',    'House-buying reason: low prices minus high prices',                 'net percent'),
    ('homrn_nr',    'House-buying reason: low rates minus high rates',                   'net percent'),
    ('homval_r',    'Home value vs a year ago (up minus down)',                          'relative score'),
    ('hompx1_r50',  'Expected home price change next 12 months: interquartile range',    'percent'),
    ('hompx5_r50',  'Expected annual home price change next 5 years: interquartile range','percent'),
    ('veh_r',       'Good vs bad time to buy a vehicle',                                 'relative score'),
    ('vehrn_np',    'Vehicle-buying reason: low prices minus high prices',               'net percent'),
    ('vehrn_nr',    'Vehicle-buying reason: low rates minus high rates',                 'net percent'),
    ('gas1px_mean', 'Expected gas price change next 12 months: mean',                    'cents per gallon'),
    ('gas1px_md',   'Expected gas price change next 12 months: median',                  'cents per gallon'),
    ('gas5px_mean', 'Expected gas price change next 5 years: mean',                      'cents per gallon'),
    ('gas5px_md',   'Expected gas price change next 5 years: median',                    'cents per gallon')
  AS t(measure_code, measure_label, unit)
),

-- Lookup: 16 monthly subgroups (codebook "Demographics" page; monthly definitions, not quarterly/yearly).
subgroups AS (
  SELECT * FROM VALUES
    ('all',   'All',        'All respondents'),
    ('a1834', 'Age',        '18-34'),
    ('a3554', 'Age',        '35-54'),
    ('a5597', 'Age',        '55 and over'),
    ('w',     'Region',     'West'),
    ('nc',    'Region',     'North Central (Midwest)'),
    ('ne',    'Region',     'Northeast'),
    ('s',     'Region',     'South'),
    ('m',     'Sex',        'Male'),
    ('f',     'Sex',        'Female'),
    ('y13',   'Income',     'Bottom third'),
    ('y23',   'Income',     'Middle third'),
    ('y33',   'Income',     'Top third'),
    ('ehs',   'Education',  'High school or less'),
    ('esc',   'Education',  'Some college'),
    ('ecd',   'Education',  'College degree or more')
  AS t(subgroup_code, subgroup_dimension, subgroup_label)
)

SELECT
  p.month,
  p.measure_code,
  m.measure_label,
  m.unit,
  p.subgroup_code,
  g.subgroup_dimension,
  g.subgroup_label,
  p.value,
  p.variable_name          -- original column name, kept for traceability back to the CSV
FROM parsed AS p
LEFT JOIN measures  AS m ON p.measure_code  = m.measure_code   -- LEFT JOIN: an unknown code shows up as a NULL label (tested in QA)
LEFT JOIN subgroups AS g ON p.subgroup_code = g.subgroup_code
WHERE p.value IS NOT NULL;

## 7. `stg_inflation` — monthly year-over-year CPI inflation, 2000–2026
**Grain:** one row per month (320 rows). The plan's headline "actual inflation" benchmark (RQ1 fundamentals table,
RQ2 expected-vs-actual).
* **October 2025 has no row** (the government shutdown month: BLS did not publish October CPI). It is left absent on
  purpose. Never interpolate it silently; any model that needs it must fill it explicitly and label the fill.
* The README does not say whether this is seasonally adjusted (a listed data gap); it matches the all-items CPI
  year-over-year change within 0.2 points.

In [0]:
CREATE OR REPLACE TABLE stg_inflation
COMMENT 'Headline CPI inflation, year-over-year percent, monthly Jan 2000 - Sep 2026. October 2025 absent (not published). Adjustment status unconfirmed.'
AS
WITH source AS (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/public_opinion_economy_case_study/raw_data/monthly_inflation_rates_2000_2026.csv',
    format => 'csv', header => true, inferColumnTypes => false
  )
)
SELECT
  to_date(`date`)                        AS month,
  try_cast(inflation_rate AS DOUBLE)     AS inflation_yoy_pct     -- e.g. 2.7 = 2.7% over the same month a year earlier
FROM source;

## 8. `stg_cpi` — BLS CPI components and real earnings
**Grain:** one row per month (129 rows, Jan 2016 – Sep 2026). Columns are **index levels** (1982–84 = 100,
seasonally adjusted; `CUSR` series), not rates. Year-over-year changes are computed in `int_econ_monthly`.
* **October 2025** exists as a row but contains `-` for all items, core, shelter and food at home, and blanks for
  earnings; `try_cast` turns these into NULL. Gasoline alone has a value that month. When computing YoY with
  `LAG`, lag by date (12 months), not by row, so the gap cannot shift months.
* **`CES0500000032` is real average hourly earnings of *production and nonsupervisory* employees** (1982–84 dollars),
  not all employees. The column name says so, and it is the right "am I keeping up?" measure for typical workers.

In [0]:
CREATE OR REPLACE TABLE stg_cpi
COMMENT 'BLS CPI-U index levels (SA, 1982-84=100) for all items, core, food at home, gasoline, shelter, plus real avg hourly earnings (production/nonsupervisory). Monthly 2016-2026.'
AS
WITH source AS (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/public_opinion_economy_case_study/raw_data/cpi_data.csv',
    format => 'csv', header => true, inferColumnTypes => false
  )
)
SELECT
  to_date(month)                                                     AS month,
  try_cast(all_items_CUSR0000SA0                     AS DOUBLE)      AS cpi_all_items,        -- headline
  try_cast(all_items_less_foodenergy_CUSR0000SA0L1E  AS DOUBLE)      AS cpi_core,             -- excl. food and energy
  try_cast(food_at_home_CUSR0000SAF11                AS DOUBLE)      AS cpi_food_at_home,     -- "visible" price: groceries
  try_cast(gasoline_all_types_CUSR0000SETB01         AS DOUBLE)      AS cpi_gasoline,         -- "visible" price: gas
  try_cast(shelter_all_customers_CUSR0000SAH1        AS DOUBLE)      AS cpi_shelter,
  try_cast(average_hourly_earnings_198284dollars_CES0500000032 AS DOUBLE)
                                                                     AS real_avg_hourly_earnings_prod_nonsup  -- 1982-84 dollars
FROM source;

## 9. `stg_misery_index`
**Grain:** one row per month (124 rows, Jun 2016 – Sep 2026). Source stores a decimal (`0.0591`) and a
`MM/YYYY` month; this model converts to percentage points (`5.91`) so it subtracts cleanly from `inflation_yoy_pct`
when `int_econ_monthly` backs out unemployment.
* **October 2025 has a value (7.29) even though October 2025 CPI does not exist** and BLS also did not publish an
  October unemployment rate. The source must have filled it, so it is flagged and should be excluded from the
  backed-out unemployment series (one more reason to add official `UNRATE`, already a listed gap).

In [0]:
CREATE OR REPLACE TABLE stg_misery_index
COMMENT 'Misery index (unemployment rate + annual inflation), monthly Jun 2016 - Sep 2026, in percentage points. Oct 2025 value flagged as filled by source.'
AS
WITH source AS (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/public_opinion_economy_case_study/raw_data/misery_index.csv',
    format => 'csv', header => true, inferColumnTypes => false
  )
)
SELECT
  to_date(month, 'MM/yyyy')                           AS month,             -- '06/2016' -> 2016-06-01
  ROUND(try_cast(misery_index AS DOUBLE) * 100, 4)    AS misery_index_pct,  -- 0.0591 -> 5.91
  to_date(month, 'MM/yyyy') = DATE'2025-10-01'        AS is_filled_by_source  -- shutdown month: no official inputs exist
FROM source;

## 10. `stg_fred_observations` — all FRED-format series in one long table
**Grain:** one row per series × observation date. **Rows:** 8,621 non-missing observations from 11 files.

Every FRED download has the same two-column shape (`observation_date`, `<SERIES_ID>`), so instead of 11
near-identical models this one stacks them and attaches metadata (frequency, units, adjustment) from a small lookup.

**Blank values are dropped, not filled.** They mean different things by series: market holidays for the stock
indices, and months between quarterly releases for the auto-loan and credit-card rates (values only in Feb, May,
Aug and Nov). In long format, *no row = no observation*, which is the honest representation. The plan's rule stands:
never forward-fill silently. The one genuine missing value (Q4 2025 real weekly earnings, a shutdown quarter) drops
out the same way.

Notes for downstream use: keep monthly `PSAVERT` as the saving-rate measure (the quarterly `A072RC1Q156SBEA` is the
same concept); stock indices are price indices, not total return; PCE and saving are revised by BEA, so record the
download vintage in `METHODS.md`.

In [0]:
CREATE OR REPLACE TABLE stg_fred_observations
COMMENT 'Long table of FRED series (rates, stock indices, saving, real PCE, real earnings, debt service). One row per series x date; blank observations dropped, never filled.'
AS
WITH observations AS (
  -- Each block: tag the series, cast the date and value. Values like '' or '.' become NULL via try_cast.
  SELECT 'MORTGAGE30US' AS series_id, to_date(observation_date) AS observation_date, try_cast(MORTGAGE30US AS DOUBLE) AS value
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/30year_fixed_mortgage_rate.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
  UNION ALL
  SELECT 'TERMCBAUTO48NS', to_date(observation_date), try_cast(TERMCBAUTO48NS AS DOUBLE)
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/48_month_auto_loan_rates.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
  UNION ALL
  SELECT 'TERMCBCCALLNS', to_date(observation_date), try_cast(TERMCBCCALLNS AS DOUBLE)
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/commercial_interest_rate_on_credit_card_plans.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
  UNION ALL
  SELECT 'DJIA', to_date(observation_date), try_cast(DJIA AS DOUBLE)
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/DJIA.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
  UNION ALL
  SELECT 'NASDAQCOM', to_date(observation_date), try_cast(NASDAQCOM AS DOUBLE)
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/NASDAQCOM.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
  UNION ALL
  SELECT 'SP500', to_date(observation_date), try_cast(SP500 AS DOUBLE)
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/SP500.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
  UNION ALL
  SELECT 'LES1252881600Q', to_date(observation_date), try_cast(LES1252881600Q AS DOUBLE)
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/employed_fulltime_median_usual_weekly_real_earnings.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
  UNION ALL
  SELECT 'TDSP', to_date(observation_date), try_cast(TDSP AS DOUBLE)
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/household_debt_service_payments_as_percent_of_disposable_personal_income.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
  UNION ALL
  SELECT 'A072RC1Q156SBEA', to_date(observation_date), try_cast(A072RC1Q156SBEA AS DOUBLE)
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/personal_saving_as_percent_of_disposable_income.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
  UNION ALL
  SELECT 'PSAVERT', to_date(observation_date), try_cast(PSAVERT AS DOUBLE)
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/personal_saving_rate.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
  UNION ALL
  SELECT 'PCEC96', to_date(observation_date), try_cast(PCEC96 AS DOUBLE)
  FROM read_files('/Volumes/workspace/public_opinion_economy_case_study/raw_data/real_personal_cons_exp.csv',
                  format => 'csv', header => true, inferColumnTypes => false)
),

-- Series metadata from the FRED series pages (README links). One place to look up what a number means.
series AS (
  SELECT * FROM VALUES
    ('MORTGAGE30US',    '30-year fixed mortgage rate (Freddie Mac)',                       'weekly',    'percent',                              'NSA'),
    ('TERMCBAUTO48NS',  'New auto loan rate, 48-month, commercial banks',                  'quarterly', 'percent',                              'NSA'),  -- monthly file, values every 3rd month
    ('TERMCBCCALLNS',   'Credit card plan interest rate, all accounts',                    'quarterly', 'percent',                              'NSA'),  -- monthly file, values every 3rd month
    ('DJIA',            'Dow Jones Industrial Average',                                    'daily',     'index (price, not total return)',      'NSA'),
    ('NASDAQCOM',       'NASDAQ Composite',                                                'daily',     'index (price, not total return)',      'NSA'),
    ('SP500',           'S&P 500',                                                         'daily',     'index (price, not total return)',      'NSA'),
    ('LES1252881600Q',  'Median usual weekly real earnings, full-time wage and salary workers','quarterly','1982-84 CPI-adjusted dollars',      'SA'),
    ('TDSP',            'Household debt service payments, % of disposable personal income','quarterly', 'percent',                              'SA'),
    ('A072RC1Q156SBEA', 'Personal saving, % of disposable personal income (quarterly)',    'quarterly', 'percent',                              'SAAR'),
    ('PSAVERT',         'Personal saving rate (monthly) - primary saving measure',         'monthly',   'percent',                              'SAAR'),
    ('PCEC96',          'Real personal consumption expenditures',                          'monthly',   'billions of chained 2017 dollars',     'SAAR')
  AS t(series_id, series_name, frequency, units, seasonal_adjustment)
)

SELECT
  o.series_id,
  s.series_name,
  s.frequency,
  s.units,
  s.seasonal_adjustment,
  o.observation_date,
  o.value
FROM observations AS o
JOIN series AS s ON o.series_id = s.series_id
WHERE o.value IS NOT NULL;   -- holidays and between-release months: no row rather than a NULL or a filled value

## 11.  `ref_events` — presidential transitions (the plan's `ref_events` seed)
**Grain:** one row per presidential election–inauguration pair (7 rows, 2000–2024).

Adds what every event study needs: the event months (monthly series join on these), the **outgoing** party, and a
`party_changed` flag. The RQ1 transitions are the rows where the party changed (2001, 2009, 2017, 2021, 2025);
re-elections (2005, 2013) are natural placebos.
* The outgoing party for the first row (2000) is not in the file; it is filled as Democrat (Clinton).
* The hand-built dates were checked: all seven election dates are correct, and Jan 20 is the constitutional
  inauguration date every time. In 2013 Jan 20 fell on a Sunday, so the oath was private that day and the public
  ceremony was Jan 21; it does not change any monthly analysis.
* Midterms (needed for the House-flip placebo in 2018 and 2022) are not in this file; they get their own small seed.

In [0]:
CREATE OR REPLACE TABLE ref_events
COMMENT 'Presidential elections and inaugurations 2000-2024 with incoming/outgoing party, party-change flag and event months for event studies.'
AS
WITH source AS (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/public_opinion_economy_case_study/raw_data/inauguration_election_dates.csv',
    format => 'csv', header => true, inferColumnTypes => false
  )
),

typed AS (
  SELECT
    to_date(election_date)       AS election_date,
    to_date(inauguration_date)   AS inauguration_date,
    trim(president_name)         AS president_name,
    trim(president_party)        AS incoming_party
  FROM source
),

with_outgoing AS (
  SELECT
    *,
    -- Party of the previous president = incoming party of the previous row. First row: Clinton (Democrat).
    COALESCE(LAG(incoming_party) OVER (ORDER BY election_date), 'Democrat')   AS outgoing_party
  FROM typed
)

SELECT
  year(election_date)                                                         AS election_year,
  concat(CAST(year(election_date) AS STRING), '-',
         CAST(year(inauguration_date) AS STRING))                             AS transition_label,   -- e.g. '2024-2025' (full years, so '2000-01' cannot be misread as a date)
  election_date,
  inauguration_date,
  trunc(election_date, 'MM')                                                  AS election_month,      -- join key for monthly series
  trunc(inauguration_date, 'MM')                                              AS inauguration_month,
  president_name,
  incoming_party,
  outgoing_party,
  incoming_party <> outgoing_party                                            AS party_changed
FROM with_outgoing;

## 12. QA checks
These are the hand-computed reconciliations the plan promises ("every mart is tested against a
hand-computed cut of the raw file"); in dbt they become `tests:` in `schema.yml` (row counts, `unique`,
`not_null`, `accepted_values`) and singular tests.

**12a. Row counts vs. the raw files.** Every row should show `pass = true`.

In [0]:
WITH actual AS (
  SELECT 'stg_ces'               AS model, COUNT(*) AS n FROM stg_ces               UNION ALL
  SELECT 'stg_anes',                       COUNT(*)      FROM stg_anes              UNION ALL
  SELECT 'stg_pew',                        COUNT(*)      FROM stg_pew               UNION ALL
  SELECT 'stg_umich_party',                COUNT(*)      FROM stg_umich_party       UNION ALL
  SELECT 'stg_umich_index',                COUNT(*)      FROM stg_umich_index       UNION ALL
  SELECT 'stg_inflation',                  COUNT(*)      FROM stg_inflation         UNION ALL
  SELECT 'stg_cpi',                        COUNT(*)      FROM stg_cpi               UNION ALL
  SELECT 'stg_misery_index',               COUNT(*)      FROM stg_misery_index      UNION ALL
  SELECT 'stg_fred_observations',          COUNT(*)      FROM stg_fred_observations UNION ALL
  SELECT 'ref_events',                     COUNT(*)      FROM ref_events
),
expected AS (
  -- Counted directly from the raw files in the GitHub repo.
  SELECT * FROM VALUES
    ('stg_ces', 718955), ('stg_anes', 73745), ('stg_pew', 28), ('stg_umich_party', 113),
    ('stg_umich_index', 92158), ('stg_inflation', 320), ('stg_cpi', 129), ('stg_misery_index', 124),
    ('stg_fred_observations', 8621), ('ref_events', 7)
  AS t(model, expected_n)
)
SELECT a.model, a.n, e.expected_n, a.n = e.expected_n AS pass
FROM actual a JOIN expected e USING (model)
ORDER BY model;

**12b. Grain, types and labels.** Each check returns the number of offending rows; every value should be **0**.


In [0]:
SELECT 'stg_ces: duplicate respondent_key'            AS check, COUNT(*) - COUNT(DISTINCT respondent_key) AS failures FROM stg_ces
UNION ALL SELECT 'stg_anes: duplicate respondent_key',          COUNT(*) - COUNT(DISTINCT respondent_key) FROM stg_anes
UNION ALL SELECT 'stg_umich_party: duplicate month',            COUNT(*) - COUNT(DISTINCT month)          FROM stg_umich_party
UNION ALL SELECT 'stg_inflation: duplicate month',              COUNT(*) - COUNT(DISTINCT month)          FROM stg_inflation
UNION ALL SELECT 'stg_cpi: duplicate month',                    COUNT(*) - COUNT(DISTINCT month)          FROM stg_cpi
UNION ALL SELECT 'stg_misery_index: duplicate month',           COUNT(*) - COUNT(DISTINCT month)          FROM stg_misery_index
UNION ALL SELECT 'stg_umich_index: duplicate month/measure/subgroup',
                 COUNT(*) - COUNT(DISTINCT month, measure_code, subgroup_code)                            FROM stg_umich_index
UNION ALL SELECT 'stg_fred_observations: duplicate series/date',
                 COUNT(*) - COUNT(DISTINCT series_id, observation_date)                                   FROM stg_fred_observations
UNION ALL SELECT 'stg_umich_index: unlabeled measure or subgroup', COUNT(*) FROM stg_umich_index WHERE measure_label IS NULL OR subgroup_label IS NULL
UNION ALL SELECT 'stg_ces: code present but label missing',     COUNT(*) FROM stg_ces
          WHERE (pid3_leaner_code IS NOT NULL AND party3 IS NULL) OR (economy_retro_code IS NOT NULL AND economy_retro IS NULL)
UNION ALL SELECT 'stg_ces: weight missing or non-positive, 2016-2025', COUNT(*) FROM stg_ces WHERE survey_year >= 2016 AND (weight IS NULL OR weight <= 0)  -- 141 zero weights exist, all in 2010
UNION ALL SELECT 'stg_anes: unexpected party3 code',            COUNT(*) FROM stg_anes WHERE party3_code NOT IN (1, 2, 3)
UNION ALL SELECT 'stg_cpi: NULL all-items outside Oct 2025',    COUNT(*) FROM stg_cpi WHERE cpi_all_items IS NULL AND month <> DATE'2025-10-01'
UNION ALL SELECT 'stg_inflation: unexpected missing months (expect only Oct 2025)',
                 CAST(months_between(MAX(month), MIN(month)) + 1 - COUNT(*) - 1 AS BIGINT)                FROM stg_inflation;

**12c. Reproduce the plan's preliminary CES numbers from `stg_ces`.** Weighted % saying the economy got better,
by party, fall 2024 vs. fall 2025.

| | 2024 incl. "not sure" | 2025 incl. | 2024 excl. "not sure" | 2025 excl. |
|---|---|---|---|---|
| Democrat | 50.8 | 5.1 | 52.5 | 5.1 |
| Republican | 4.9 | 50.8 | 5.0 | 52.2 |
| Independent | 10.9 | 12.2 | 11.7 | 13.1 |

The plan's preliminary numbers (Rep 4.9→50.8, Dem 50.8→5.1, Ind 10.9→12.2) match the **"incl. not sure"** column,
but the plan's main specification says "not sure" is **excluded**. Pick one before the presentation: either update
the reported numbers to the excluded version, or state that the descriptive cut keeps "not sure" in the base.

In [0]:
SELECT
  survey_year,
  party3,
  ROUND(100 * SUM(IF(economy_retro_code IN (1, 2), weight, 0)) / SUM(weight), 1)                         AS pct_better_incl_not_sure,
  ROUND(100 * SUM(IF(economy_retro_code IN (1, 2), weight, 0)) / SUM(IF(economy_retro_code <> 6, weight, 0)), 1)
                                                                                                         AS pct_better_excl_not_sure,
  COUNT(*)                                                                                               AS n_respondents
FROM stg_ces
WHERE survey_year IN (2024, 2025)
  AND party3 IN ('Democrat', 'Republican', 'Independent')
  AND economy_retro_code IS NOT NULL
GROUP BY survey_year, party3
ORDER BY party3, survey_year;

**12d. Reproduce the plan's ANES RQ3 table from `stg_anes` + `ref_events`.** Gap = weighted % "worse" among the
out-party minus the in-party. The White House party for each survey year comes from `ref_events` (latest
inauguration before the fall of that year), which also tests the events table. "Do not know" stays in the base, as in
the plan.

Expected: 2016 → 29.1 / 12.8 · 2020 → 39.3 / 11.0 · 2024 → 50.3 / 25.3. The plan shows 11.1 for 2020's personal
gap; the 0.1 difference is because the first pass kept code 0 (NA / no post-election interview) in the base, which
staging now sets to NULL.

In [0]:
%sql
WITH white_house AS (
  -- Party holding the White House during each ANES survey year.
  SELECT a.survey_year,
         MAX_BY(e.incoming_party, e.inauguration_date) AS in_party
  FROM (SELECT DISTINCT survey_year FROM stg_anes WHERE survey_year IN (2016, 2020, 2024)) a
  JOIN ref_events e ON e.inauguration_date <= make_date(a.survey_year, 9, 1)
  GROUP BY a.survey_year
),
pct_worse AS (
  SELECT
    s.survey_year,
    w.in_party,
    s.party3,
    100 * SUM(IF(s.econ_national_retro = 'Worse', s.weight, 0)) / SUM(IF(s.econ_national_retro IS NOT NULL, s.weight, 0)) AS pct_worse_national,
    100 * SUM(IF(s.personal_fin_retro  = 'Worse', s.weight, 0)) / SUM(IF(s.personal_fin_retro  IS NOT NULL, s.weight, 0)) AS pct_worse_personal
  FROM stg_anes s
  JOIN white_house w USING (survey_year)
  WHERE s.party3 IN ('Democrat', 'Republican')
  GROUP BY s.survey_year, w.in_party, s.party3
)
SELECT
  survey_year,
  in_party                                                                                 AS party_in_white_house,
  ROUND(MAX(IF(party3 <> in_party, pct_worse_national, NULL)) - MAX(IF(party3 = in_party, pct_worse_national, NULL)), 1) AS national_gap_pts,
  ROUND(MAX(IF(party3 <> in_party, pct_worse_personal, NULL)) - MAX(IF(party3 = in_party, pct_worse_personal, NULL)), 1) AS personal_gap_pts
FROM pct_worse
GROUP BY survey_year, in_party
ORDER BY survey_year;

**12e. Spot-check the long UMich table against a number in the plan.** Median expected inflation peaked at **6.6% in
May 2025**; this should return 6.6 for May 2025 and show the rise around it.

In [0]:
%sql
SELECT month, value AS px1_median_expected_inflation_pct
FROM stg_umich_index
WHERE measure_code = 'px1_med' AND subgroup_code = 'all'
  AND month BETWEEN DATE'2025-01-01' AND DATE'2025-08-01'
ORDER BY month;